# County sociodemographics (ACS 5-year) - per-year collection 2010-2024

The TWFE panel spans 2010-2025, so each county needs sociodemographic controls
that **vary by year**.  This notebook pulls the American Community Survey 5-year
(ACS5) county tables for California from the Census API once **per year** and
stacks them into a long `county_name x year` file.

ACS5 is published through 2024 (the 2025 5-year release is not out yet), so we
collect 2010-2024; `01_build_panel.sql` carries the latest available year (2024)
forward to the panel's 2025 row.  Same source as the original single-year pull,
so the series is internally consistent.

In [1]:
import os
import requests
import pandas as pd

In [2]:
CENSUS_KEY = "77e29a2ecf656c41fb214433c56d1f57dd6a6d0a"

# ACS5 vintages to collect (2010-2024 inclusive).  2025 is not yet released.
YEARS = list(range(2010, 2025))

variables = [
    "B19013_001E",  # median HH income
    "B01003_001E",  # total population
    "B03002_003E",  # White alone (non-Hispanic)
    "B03002_012E",  # Hispanic or Latino
    "B03002_004E",  # Black/AA
    "B25003_001E", "B25003_002E",  # housing tenure
    "B08301_001E", "B08301_002E",  # commute mode
    # Income brackets for OBBBA exposure intensity
    "B19001_001E","B19001_002E","B19001_003E","B19001_004E",
    "B19001_005E","B19001_006E","B19001_007E","B19001_008E",
    "B19001_009E","B19001_010E","B19001_011E","B19001_012E",
    "B19001_013E","B19001_014E","B19001_015E","B19001_016E","B19001_017E",
]
get_str = "NAME," + ",".join(variables)

In [3]:
rename_map = {
    # Geography
    "NAME"          : "county_name",
    "state"         : "state_fips",
    "county"        : "county_fips",
    # Median household income
    "B19013_001E"   : "median_hh_income",
    # Total population
    "B01003_001E"   : "total_population",
    # Race / ethnicity
    "B03002_003E"   : "pop_white_non_hispanic",
    "B03002_004E"   : "pop_black_non_hispanic",
    "B03002_012E"   : "pop_hispanic_any_race",
    # Housing tenure
    "B25003_001E"   : "housing_units_total",
    "B25003_002E"   : "housing_units_owner_occupied",
    # Commute mode
    "B08301_001E"   : "commuters_total",
    "B08301_002E"   : "commuters_drove_alone",
    # Household income brackets (counts of households)
    "B19001_001E"   : "hh_income_total",
    "B19001_002E"   : "hh_income_under_10k",
    "B19001_003E"   : "hh_income_10k_to_14999",
    "B19001_004E"   : "hh_income_15k_to_19999",
    "B19001_005E"   : "hh_income_20k_to_24999",
    "B19001_006E"   : "hh_income_25k_to_29999",
    "B19001_007E"   : "hh_income_30k_to_34999",
    "B19001_008E"   : "hh_income_35k_to_39999",
    "B19001_009E"   : "hh_income_40k_to_44999",
    "B19001_010E"   : "hh_income_45k_to_49999",
    "B19001_011E"   : "hh_income_50k_to_59999",
    "B19001_012E"   : "hh_income_60k_to_74999",
    "B19001_013E"   : "hh_income_75k_to_99999",
    "B19001_014E"   : "hh_income_100k_to_124999",
    "B19001_015E"   : "hh_income_125k_to_149999",
    "B19001_016E"   : "hh_income_150k_to_199999",
    "B19001_017E"   : "hh_income_200k_and_over",
}

# Numeric estimate columns (everything except the geography strings).
est_cols = [v for v in variables]
income_cols  = [f"B19001_{str(i).zfill(3)}E" for i in range(1, 18)]   # 001-017
under_150k   = [f"B19001_{str(i).zfill(3)}E" for i in range(2, 16)]   # 002-015 (< $150k)

In [4]:
def fetch_year(year: int) -> pd.DataFrame:
    """Pull the ACS5 county table for one vintage year."""
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": get_str,
        "for": "county:*",
        "in": "state:06",
        "key": CENSUS_KEY,
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    rows = r.json()
    df = pd.DataFrame(rows[1:], columns=rows[0])

    # Census uses large negative sentinels (e.g. -666666666) for missing /
    # suppressed estimates - coerce them to NaN before any arithmetic.
    for c in est_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c] < 0, c] = pd.NA

    # OBBBA exposure proxy: share of households earning under $150k.
    df["share_under_150k"] = df[under_150k].sum(axis=1) / df["B19001_001E"]

    df = df.rename(columns=rename_map)
    df["year"] = year
    return df

In [5]:
OUT_DIR = "acs_by_year"
os.makedirs(OUT_DIR, exist_ok=True)

frames = []
for yr in YEARS:
    dfy = fetch_year(yr)
    dfy.to_csv(os.path.join(OUT_DIR, f"census_acs_county_{yr}.csv"), index=False)
    frames.append(dfy)
    print(f"{yr}: {len(dfy)} counties")

acs = pd.concat(frames, ignore_index=True)

2010: 58 counties
2011: 58 counties
2012: 58 counties
2013: 58 counties
2014: 58 counties
2015: 58 counties
2016: 58 counties
2017: 58 counties
2018: 58 counties
2019: 58 counties
2020: 58 counties
2021: 58 counties
2022: 58 counties
2023: 58 counties
2024: 58 counties


In [6]:
# Order columns: geography, year, then the renamed measures.
ordered = ["county_name", "state_fips", "county_fips", "year"] + \
          [rename_map[v] for v in variables] + ["share_under_150k"]
acs = acs[ordered]

# Sanity check: no raw Census B-codes should remain.
leftover = [c for c in acs.columns if c.startswith("B") and c.endswith("E")]
print("Unrenamed Census codes:" , leftover if leftover else "none")
print(f"Rows: {len(acs):,}  Years: {acs['year'].nunique()}  Counties/yr: {acs.groupby('year').size().unique()}")

Unrenamed Census codes: none
Rows: 870  Years: 15  Counties/yr: [58]


In [7]:
# Quick look: median income should change across years for a given county.
print(acs[acs.county_fips == "001"][["year", "median_hh_income", "total_population"]].to_string(index=False))

 year  median_hh_income  total_population
 2010           69384.0         1477980.0
 2011           70821.0         1494876.0
 2012           71516.0         1515136.0
 2013           72112.0         1535248.0
 2014           73775.0         1559308.0
 2015           75619.0         1584983.0
 2016           79831.0         1605217.0
 2017           85743.0         1629615.0
 2018           92574.0         1643700.0
 2019           99406.0         1656754.0
 2020          104888.0         1661584.0
 2021          112017.0         1673133.0
 2022          122488.0         1663823.0
 2023          126240.0         1651949.0
 2024          129367.0         1649473.0


In [8]:
# Consolidated long file consumed by 01_build_panel.sql (BULK INSERT target).
acs.to_csv("census_acs_county_data.csv", index=False)
print("Wrote census_acs_county_data.csv:", acs.shape)

Wrote census_acs_county_data.csv: (870, 31)
